In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots

In [ ]:
file_515 = r"J:\ctgroup\Edward\DATA\VMI\20250408\calibrated\po_500mW_ell_calibrated.h5"
file_1030 = r"J:\ctgroup\Edward\DATA\VMI\20250307\Intensity Scan\4_calibrated.h5"

df_515 = pd.read_hdf(file_515)
df_1030 = pd.read_hdf(file_1030)
pmax = 0.6
df_515 = df_515[(df_515['px'] ** 2 + df_515['py'] ** 2 + df_515['pz'] ** 2) < pmax ** 2]
df_1030 = df_1030[(df_1030['px'] ** 2 + df_1030['py'] ** 2 + df_1030['pz'] ** 2) < pmax ** 2]

mq_range = (0, 70)
df_515 = df_515[(df_515['m/q'] > mq_range[0]) & (df_515['m/q'] < mq_range[1])]
df_1030 = df_1030[(df_1030['m/q'] > mq_range[0]) & (df_1030['m/q'] < mq_range[1])]

df_515['pr'] = np.sqrt(df_515['px'] ** 2 + df_515['py'] ** 2 + df_515['pz'] ** 2)
df_1030['pr'] = np.sqrt(df_1030['px'] ** 2 + df_1030['py'] ** 2 + df_1030['pz'] ** 2)

In [ ]:
gates = {
    "C2H4+": (20, 33),
    "C2H3O+": (36, 47),
    "Parent PO+": (50, 60),
}

px.histogram(df_515, x='m/q', nbins=500, title='515nm Mass Spectrum', labels={'m/q': 'm/q (a.u.)'}).show()
px.histogram(df_1030, x='m/q', nbins=500, title='1030nm Mass Spectrum', labels={'m/q': 'm/q (a.u.)'}).show()

In [ ]:
for name, (low, high) in gates.items():
    df_515.loc[(df_515['m/q'] > low) & (df_515['m/q'] < high), 'ion'] = name
    df_1030.loc[(df_1030['m/q'] > low) & (df_1030['m/q'] < high), 'ion'] = name
r_grid = np.linspace(0.001, 0.5, 1000)
hists_515 = [np.histogram(df_515[df_515['ion'] == i]['pr'], bins=1000, range=(0, 0.5))[0] / r_grid for i in
             gates.keys()]
hists_1030 = [np.histogram(df_1030[df_1030['ion'] == i]['pr'], bins=1000, range=(0, 0.5))[0] / r_grid for i in
              gates.keys()]
fig = make_subplots(2, 1, subplot_titles=("515nm", "1030nm"), shared_xaxes=True, x_title="pr (a.u.)",
                    y_title="Signal (arb. units)")
[fig.add_scatter(
        x=r_grid,
        y=hist_515 / np.max(hist_515),
        line_color=color,
        name=f"{name}",
        row=1, col=1,
) for hist_515, name, color in zip(hists_515, gates.keys(), ['red', 'green', 'blue'])]
[fig.add_scatter(
        x=r_grid,
        y=hist_1030 / np.max(hist_1030),
        line_color=color,
        row=2, col=1,
        showlegend=False,
) for hist_1030, name, color in zip(hists_1030, gates.keys(), ['red', 'green', 'blue'])]
fig.update_layout(
        width=800,
        height=800,
        template='simple_white+presentation',
        legend_orientation="h",
        legend_x=0,
        legend_y=1.1,
)
fig.show()
